In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/train.csv')  # adjust filename if different
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (891, 12)

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Null counts:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate rows: 0


In [3]:
# Age: numeric, moderate nulls -> median imputation (robust to outliers)
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked: categorical, very few nulls -> mode imputation
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Cabin: too many nulls to impute meaningfully -> drop the column
df.drop('Cabin', axis=1, inplace=True)

print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


C:\Users\ASHUTOSH\AppData\Local\Temp\ipykernel_28100\901838280.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\ASHUTOSH\AppData\Local\Temp\ipykernel_28100\901838280.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [4]:
before_rows = len(df)
df.drop_duplicates(inplace=True)
after_rows = len(df)
print(f"Removed {before_rows - after_rows} duplicate rows")

Removed 0 duplicate rows


In [5]:
df['Sex'] = df['Sex'].str.strip().str.capitalize()
df['Embarked'] = df['Embarked'].str.strip().str.upper()
print(df['Sex'].unique())
print(df['Embarked'].unique())

['Male' 'Female']
['S' 'C' 'Q']


In [6]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df[(df['Fare'] < lower) | (df['Fare'] > upper)]
print(f"Number of Fare outliers: {len(outliers)}")

# Decision: cap extreme values instead of removing (preserves sample size)
df['Fare'] = df['Fare'].clip(lower=lower, upper=upper)

Number of Fare outliers: 116


In [7]:
df['PassengerId'] = df['PassengerId'].astype(str)
df['Survived'] = df['Survived'].astype(int)
df['Pclass'] = df['Pclass'].astype(int)
print(df.dtypes)

PassengerId     object
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object


In [8]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Total Nulls', 'Duplicate Rows'],
    'Before': [before_rows, "See Cell 2 output", "See Cell 2 output"],
    'After': [after_rows, df.isnull().sum().sum(), df.duplicated().sum()]
})
summary


,Metric,Before,After
0,Rows,891,891
1,Total Nulls,See Cell 2 output,0
2,Duplicate Rows,See Cell 2 output,0


In [9]:
df.to_csv('data/cleaned_titanic.csv', index=False)
print("Saved cleaned dataset.")

Saved cleaned dataset.
